# Qom–Spanish MT Baseline (NLLB-600M)

Direction: QOM→ES, with stratified splits.

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install sacrebleu evaluate openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # single GPU — avoids DataParallel OOM on T4×2

import gc, re, random, unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Optional, Tuple

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Paths
DATA_DIR = Path("/kaggle/input/datasets/alexkorablev/qom-corpus")
OUT_DIR  = Path("/kaggle/working/qom-nllb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Splits — Copaic held out as out-of-domain test
HOLDOUT_KEYWORDS = {"copaic": "test"}
TRAIN_FRAC = 0.80
DEV_FRAC   = 0.10
TEST_FRAC  = 0.10

# Views
SENT_MAX_CHARS = 260
SENT_MIN_CHARS = 20
TRAIN_VIEW = "line"   # most data; change to "sent" or "frag" to experiment

# Model
MODEL_NAME     = "facebook/nllb-200-distilled-600M"
SRC_LANG       = "spa_Latn"
TGT_LANG = "grn_Latn"
MAX_LEN        = 128

# Training
NUM_EPOCHS  = 10
TRAIN_BATCH = 2
EVAL_BATCH  = 2
GRAD_ACCUM  = 8    # effective batch = 16

print("Config OK")
print(f"  DATA_DIR  : {DATA_DIR}")
print(f"  OUT_DIR   : {OUT_DIR}")
print(f"  MODEL     : {MODEL_NAME}")
print(f"  TRAIN_VIEW: {TRAIN_VIEW}")

Config OK
  DATA_DIR  : /kaggle/input/datasets/alexkorablev/qom-corpus
  OUT_DIR   : /kaggle/working/qom-nllb
  MODEL     : facebook/nllb-200-distilled-600M
  TRAIN_VIEW: line


In [3]:
# ── Cell 3: Load xlsx corpus + metadata name pairs ────────────────────────────
def read_workbook(path: Path) -> pd.DataFrame:
    doc_name = path.stem
    xl = pd.ExcelFile(path)
    frames = []
    for sheet in xl.sheet_names:
        try:
            raw = xl.parse(sheet)
        except Exception as e:
            print(f"  [SKIP] {sheet}: {e}")
            continue
        raw.columns = [str(c).strip() for c in raw.columns]
        if "linea_qom" not in raw.columns or "linea_es" not in raw.columns:
            continue
        sub = raw[["linea_qom", "linea_es"]].copy()
        sub.columns = ["qom", "es"]
        sub = sub.dropna(subset=["qom", "es"])
        sub["qom"] = sub["qom"].astype(str).str.strip()
        sub["es"]  = sub["es"].astype(str).str.strip()
        sub = sub[(sub["qom"] != "") & (sub["es"] != "")]
        sub = sub[(sub["qom"] != "nan") & (sub["es"] != "nan")]
        for meta_col in ["id_linea", "id_fragmento", "id_seccion", "id_capitulo"]:
            sub[meta_col] = raw[meta_col].values[:len(sub)] if meta_col in raw.columns else np.nan
        sub["source_doc"]  = doc_name
        sub["source_path"] = str(path)
        sub["sheet_name"]  = sheet
        frames.append(sub)
    if not frames:
        print(f"  [WARN] No usable sheets in {path.name}")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def read_metadata_names(path: Path) -> pd.DataFrame:
    """Extract qom/es name pairs from metadata tabs (Capítulos, Secciones, Fragmentos)."""
    doc_name = path.stem
    xl = pd.ExcelFile(path)
    pairs = []
    for sheet in xl.sheet_names:
        try:
            raw = xl.parse(sheet)
        except Exception:
            continue
        raw.columns = [str(c).strip() for c in raw.columns]
        # Skip the main Lineas tab
        if "linea_qom" in raw.columns:
            continue
        # Find paired qom/es name columns
        qom_cols = [c for c in raw.columns if c.startswith("nombre_") and c.endswith("_qom")]
        for qc in qom_cols:
            ec = qc.replace("_qom", "_es")
            if ec not in raw.columns:
                continue
            sub = raw[[qc, ec]].dropna().copy()
            sub.columns = ["qom", "es"]
            sub["qom"] = sub["qom"].astype(str).str.strip()
            sub["es"]  = sub["es"].astype(str).str.strip()
            sub = sub[(sub["qom"] != "") & (sub["es"] != "")]
            sub = sub[(sub["qom"] != "nan") & (sub["es"] != "nan")]
            sub = sub[(sub["qom"].str.lower() != "n/a") & (sub["es"].str.lower() != "n/a")]
            if len(sub) > 0:
                sub["source_doc"] = doc_name
                sub["source_path"] = str(path)
                sub["sheet_name"] = sheet
                sub["id_linea"] = [f"{doc_name.lower().replace(' ','_')}_meta_{sheet.lower()}_{i}" for i in range(len(sub))]
                sub["id_fragmento"] = np.nan
                sub["id_seccion"] = np.nan
                sub["id_capitulo"] = np.nan
                pairs.append(sub)
                print(f"  [META] {sheet}: {qc} / {ec} → {len(sub)} name pairs")
    if not pairs:
        return pd.DataFrame()
    return pd.concat(pairs, ignore_index=True)

xlsx_files = sorted(DATA_DIR.rglob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError(f"No .xlsx files found under {DATA_DIR}")
print(f"Found {len(xlsx_files)} xlsx files:")
for p in xlsx_files:
    print(" -", p.name)

frames = []
meta_frames = []
for p in xlsx_files:
    print(f"Reading {p.name} …")
    f = read_workbook(p)
    if len(f):
        frames.append(f)
        print(f"  → {len(f)} line pairs")
    m = read_metadata_names(p)
    if len(m):
        meta_frames.append(m)

df = pd.concat(frames, ignore_index=True)
if meta_frames:
    meta_df = pd.concat(meta_frames, ignore_index=True)
    print(f"\nMetadata name pairs: {len(meta_df)}")
    df = pd.concat([df, meta_df], ignore_index=True)
else:
    print("\nNo metadata name pairs found")

# Stable line IDs where missing
mask = df["id_linea"].isna()
df.loc[mask, "id_linea"] = [
    f"{row.source_doc.lower().replace(' ','_')}_{i+1}"
    for i, row in df[mask].iterrows()
]
print(f"\nTotal rows (lines + metadata): {len(df)}")
df.head(3)


Found 4 xlsx files:
 - Arte verbal qom.xlsx
 - Educacion Sanitaria Intercultural.xlsx
 - Las Aventuras de Copaic.xlsx
 - Materiales del Taller de Lengua y Cultura Toba.xlsx
Reading Arte verbal qom.xlsx …
  → 1825 line pairs
  [META] Capítulos: nombre_capitulo_qom / nombre_capitulo_es → 4 name pairs
  [META] Fragmentos: nombre_fragmento_qom / nombre_fragmento_es → 62 name pairs
Reading Educacion Sanitaria Intercultural.xlsx …
  → 169 line pairs
  [META] Capítulos: nombre_capitulo_qom / nombre_capitulo_es → 3 name pairs
  [META] Fragmentos: nombre_fragmento_qom / nombre_fragmento_es → 42 name pairs
Reading Las Aventuras de Copaic.xlsx …
  → 16 line pairs
Reading Materiales del Taller de Lengua y Cultura Toba.xlsx …
  → 267 line pairs
  [META] Secciones: nombre_seccion_qom / nombre_seccion_es → 3 name pairs
  [META] Fragmentos: nombre_fragmento_qom / nombre_fragmento_es → 8 name pairs

Metadata name pairs: 122

Total rows (lines + metadata): 2399


,qom,es,id_linea,id_fragmento,id_seccion,id_capitulo,source_doc,source_path,sheet_name
0,Lauattonaxanaxac na qom,La sabiduría de los qom,arteverbal_1,NaN,NaN,intro,Arte verbal qom,/kaggle/input/datasets/alexkorablev/qom-corpus...,Lineas
1,Cha'aye ne'ena de'eda qaỹachegoqta'ague qaica...,Porque en el origen no había nadie que enseñar...,arteverbal_2,NaN,NaN,intro,Arte verbal qom,/kaggle/input/datasets/alexkorablev/qom-corpus...,Lineas
2,qataq qaica ca lhuotta'a ca latannaxanaxanat,y tampoco existía ningún medicamento preparado...,arteverbal_3,NaN,NaN,intro,Arte verbal qom,/kaggle/input/datasets/alexkorablev/qom-corpus...,Lineas


In [4]:
# ── Cell 3b: Load CSV sources (El Principito + La Biblia) ───────────────────
def read_csv_source(path: Path, qom_col: str, es_col: str, doc_name: str) -> pd.DataFrame:
    raw = pd.read_csv(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    sub = raw[[qom_col, es_col]].copy()
    sub.columns = ["qom", "es"]
    sub = sub.dropna(subset=["qom", "es"])
    sub["qom"] = sub["qom"].astype(str).str.strip()
    sub["es"]  = sub["es"].astype(str).str.strip()
    sub = sub[(sub["qom"] != "") & (sub["es"] != "")]
    sub = sub[(sub["qom"] != "nan") & (sub["es"] != "nan")]
    sub["id_linea"]     = [f"{doc_name.lower().replace(' ','_')}_{i+1}" for i in range(len(sub))]
    sub["id_fragmento"] = np.nan
    sub["id_seccion"]   = np.nan
    sub["id_capitulo"]  = np.nan
    sub["source_doc"]   = doc_name
    sub["source_path"]  = str(path)
    sub["sheet_name"]   = "csv"
    print(f"  {doc_name}: {len(sub)} pairs")
    return sub

csv_frames = []
principito_path = DATA_DIR / "El Principito.csv"
if principito_path.exists():
    csv_frames.append(read_csv_source(principito_path, qom_col="qom", es_col="espanol", doc_name="El Principito"))
else:
    print(f"  [WARN] Not found: {principito_path}")

biblia_path = DATA_DIR / "La Biblia.csv"
if biblia_path.exists():
    csv_frames.append(read_csv_source(biblia_path, qom_col="LNLE13", es_col="DHHS94", doc_name="La Biblia"))
else:
    print(f"  [WARN] Not found: {biblia_path}")

if csv_frames:
    csv_df = pd.concat(csv_frames, ignore_index=True)
    df = pd.concat([df, csv_df], ignore_index=True)
    print(f"Total rows after adding CSV sources: {len(df)}")
else:
    print("No CSV sources loaded.")


  El Principito: 557 pairs
  La Biblia: 30580 pairs
Total rows after adding CSV sources: 33536


In [5]:
# ── Cell 4: QC filtering ──────────────────────────────────────────────────────
_tok = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def simple_tokens(s):
    return [t for t in _tok.findall(s) if t.strip()]

df["qom_len_c"]   = df["qom"].str.len()
df["es_len_c"]    = df["es"].str.len()
df["qom_len_t"]   = df["qom"].map(lambda s: len(simple_tokens(s)))
df["es_len_t"]    = df["es"].map(lambda s: len(simple_tokens(s)))
df["len_ratio_t"] = (df["es_len_t"] + 1) / (df["qom_len_t"] + 1)

before = len(df)
df = df.drop_duplicates(subset=["qom", "es", "source_doc"], keep="first").copy()
print(f"Dedup removed {before - len(df)} rows")

df = df[(df["qom"] != "") & (df["es"] != "")].copy()
df = df[(df["len_ratio_t"] > 0.15) & (df["len_ratio_t"] < 6.0)].copy()
df = df[(df["qom_len_c"] < 800) & (df["es_len_c"] < 800)].copy()

print(f"Rows after QC: {len(df)}")

Dedup removed 174 rows
Rows after QC: 33331


In [6]:
# ── Cell 5: Build views ───────────────────────────────────────────────────────
def get_order_col(d):
    if "id_linea" in d.columns:
        try:
            return pd.to_numeric(d["id_linea"], errors="coerce").fillna(np.arange(len(d)))
        except Exception:
            pass
    return pd.Series(np.arange(len(d)), index=d.index)

def punctuation_boundary(s):
    return bool(re.search(r"[.!?]\s*$", s))

def build_sentenceish(d):
    d = d.copy()
    d["order"] = get_order_col(d)
    def group_key(row):
        if pd.notna(row.get("id_fragmento")):
            return (row["source_doc"], "frag", row["id_fragmento"])
        if pd.notna(row.get("id_seccion")):
            return (row["source_doc"], "sec", row["id_seccion"])
        return (row["source_doc"], "doc", row["source_doc"])
    d["_gkey"] = d.apply(group_key, axis=1)
    records = []
    for gkey, grp in d.groupby("_gkey", sort=False):
        grp = grp.sort_values("order")
        buf_q, buf_e, first_id = [], [], None
        for _, row in grp.iterrows():
            if first_id is None:
                first_id = row["id_linea"]
            buf_q.append(row["qom"])
            buf_e.append(row["es"])
            combined = " ".join(buf_q)
            if len(combined) >= SENT_MIN_CHARS and (
                punctuation_boundary(combined) or len(combined) >= SENT_MAX_CHARS
            ):
                records.append({"source_doc": row["source_doc"], "id_linea": first_id,
                                "id_fragmento": row.get("id_fragmento"),
                                "id_seccion": row.get("id_seccion"),
                                "id_capitulo": row.get("id_capitulo"),
                                "qom": " ".join(buf_q), "es": " ".join(buf_e)})
                buf_q, buf_e, first_id = [], [], None
        if buf_q:
            records.append({"source_doc": grp.iloc[-1]["source_doc"], "id_linea": first_id,
                            "id_fragmento": grp.iloc[-1].get("id_fragmento"),
                            "id_seccion": grp.iloc[-1].get("id_seccion"),
                            "id_capitulo": grp.iloc[-1].get("id_capitulo"),
                            "qom": " ".join(buf_q), "es": " ".join(buf_e)})
    return pd.DataFrame(records)

def build_fragment(d):
    key_col = next(
        (c for c in ["id_fragmento", "id_seccion", "id_capitulo"]
         if c in d.columns and d[c].notna().any()), None)
    if key_col is None:
        return pd.DataFrame()
    records = []
    for (doc, frag_id), grp in d.groupby(["source_doc", key_col]):
        records.append({"source_doc": doc, "id_linea": grp.iloc[0]["id_linea"],
                        key_col: frag_id,
                        "qom": " ".join(grp["qom"].tolist()),
                        "es": " ".join(grp["es"].tolist())})
    return pd.DataFrame(records)

views = {
    "line": df[["source_doc", "id_linea", "id_fragmento", "id_seccion", "id_capitulo", "qom", "es"]].copy(),
    "sent": build_sentenceish(df),
    "frag": build_fragment(df),
}
for name, v in views.items():
    print(f"{name}  {len(v)} pairs")

line  33331 pairs
sent  27237 pairs
frag  135 pairs


In [7]:
# ── Cell 6: Train/dev/test split (leakage-safe, global random) ─────────────────
def assign_group_id(d):
    def _gid(row):
        if pd.notna(row.get("id_fragmento")):
            return f"{row['source_doc']}__frag__{row['id_fragmento']}"
        if pd.notna(row.get("id_seccion")):
            return f"{row['source_doc']}__sec__{row['id_seccion']}"
        if pd.notna(row.get("id_capitulo")):
            return f"{row['source_doc']}__cap__{row['id_capitulo']}"
        return f"{row['source_doc']}__line__{row['id_linea']}"
    return d.apply(_gid, axis=1)

def split_dataset(d, holdout_kws, train_frac, dev_frac, seed):
    d = d.copy()
    d["group_id"] = assign_group_id(d)

    # Pull out forced-test docs (e.g. Copaic)
    forced_test_mask = pd.Series(False, index=d.index)
    for kw in holdout_kws:
        forced_test_mask |= d["source_doc"].str.lower().str.contains(kw, na=False)
    forced_test = d[forced_test_mask]
    rest = d[~forced_test_mask]

    # Global random shuffle of groups
    groups = rest["group_id"].unique().tolist()
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_train = max(1, int(len(groups) * train_frac))
    n_dev   = max(1, int(len(groups) * dev_frac))
    train_g = set(groups[:n_train])
    dev_g   = set(groups[n_train:n_train + n_dev])
    test_g  = set(groups[n_train + n_dev:])

    train_df = rest[rest["group_id"].isin(train_g)].copy()
    dev_df   = rest[rest["group_id"].isin(dev_g)].copy()
    test_df  = pd.concat([rest[rest["group_id"].isin(test_g)], forced_test],
                         ignore_index=True).copy()
    return train_df, dev_df, test_df

selected_view = views[TRAIN_VIEW].copy()
train_df, dev_df, test_df = split_dataset(
    selected_view, HOLDOUT_KEYWORDS, TRAIN_FRAC, DEV_FRAC, RANDOM_SEED
)
print(f"Train: {len(train_df)}  Dev: {len(dev_df)}  Test: {len(test_df)}")
print(f"\nPer-doc distribution:")
for split_name, split_df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    counts = split_df["source_doc"].value_counts().to_dict()
    print(f"  {split_name}: {counts}")


Train: 26802  Dev: 3256  Test: 3273

Per-doc distribution:
  train: {'La Biblia': 24326, 'Arte verbal qom': 1592, 'El Principito': 466, 'Materiales del Taller de Lengua y Cultura Toba': 243, 'Educacion Sanitaria Intercultural': 175}
  dev: {'La Biblia': 3052, 'Arte verbal qom': 111, 'El Principito': 48, 'Materiales del Taller de Lengua y Cultura Toba': 27, 'Educacion Sanitaria Intercultural': 18}
  test: {'La Biblia': 3071, 'Arte verbal qom': 128, 'El Principito': 36, 'Educacion Sanitaria Intercultural': 19, 'Las Aventuras de Copaic': 16, 'Materiales del Taller de Lengua y Cultura Toba': 3}


In [8]:
# ── Cell 7: Leakage checks ───────────────────────────────────────────────────
def assert_no_group_leakage(train, dev, test):
    t = set(train["group_id"].unique())
    d = set(dev["group_id"].unique())
    s = set(test["group_id"].unique())
    assert t.isdisjoint(d), "LEAKAGE: train/dev share group_id(s)"
    assert t.isdisjoint(s), "LEAKAGE: train/test share group_id(s)"
    assert d.isdisjoint(s), "LEAKAGE: dev/test share group_id(s)"
    print("✅ No group-level leakage")

def assert_no_exact_pair_overlap(train, dev, test):
    def pairs(x): return set(zip(x["qom"], x["es"]))
    pt, pdv, pts = pairs(train), pairs(dev), pairs(test)
    assert pt.isdisjoint(pdv),  "LEAKAGE: train/dev share exact pairs"
    assert pt.isdisjoint(pts),  "LEAKAGE: train/test share exact pairs"
    assert pdv.isdisjoint(pts), "LEAKAGE: dev/test share exact pairs"
    print("✅ No exact-pair overlap")

assert_no_group_leakage(train_df, dev_df, test_df)
assert_no_exact_pair_overlap(train_df, dev_df, test_df)

stats = pd.DataFrame([
    {"split": s, "rows": len(d),
     "qom_len_mean": round(d["qom"].str.len().mean(), 1),
     "es_len_mean":  round(d["es"].str.len().mean(),  1)}
    for s, d in [("train", train_df), ("dev", dev_df), ("test", test_df)]
])
print(stats.to_string(index=False))
print("\n✅ All checks passed")


✅ No group-level leakage
✅ No exact-pair overlap
split  rows  qom_len_mean  es_len_mean
train 26802         182.1        122.1
  dev  3256         186.6        124.6
 test  3273         191.0        126.6

✅ All checks passed


In [9]:
# ── Cell 8: Full training + evaluation function ──────────────────────────────
import torch, gc, os, sacrebleu, shutil
from transformers import (NllbTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          DataCollatorForSeq2Seq)
from datasets import Dataset

def run_direction(direction_label, src_col, tgt_col, src_lang, tgt_lang,
                  train_df, dev_df, test_df, out_suffix):
    """Train and evaluate one translation direction end-to-end."""

    print(f"\n{'='*70}")
    print(f"  DIRECTION: {direction_label}")
    print(f"{'='*70}\n")

    run_dir = OUT_DIR / out_suffix
    run_dir.mkdir(parents=True, exist_ok=True)

    # ── Load model fresh ──
    gc.collect()
    torch.cuda.empty_cache()
    tokenizer = NllbTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.gradient_checkpointing_enable()
    model = model.to("cuda")
    print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

    # ── Build datasets ──
    def make_records(d):
        return [{"src": row[src_col], "tgt": row[tgt_col]} for _, row in d.iterrows()]

    def preprocess(example):
        tokenizer.src_lang = src_lang
        model_inputs = tokenizer(example["src"], max_length=MAX_LEN, truncation=True)
        tokenizer.src_lang = tgt_lang
        labels = tokenizer(example["tgt"], max_length=MAX_LEN, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        tokenizer.src_lang = src_lang
        return model_inputs

    def records_to_dataset(records):
        return Dataset.from_dict({"src": [r["src"] for r in records], "tgt": [r["tgt"] for r in records]})

    train_dataset = records_to_dataset(make_records(train_df)).map(preprocess, remove_columns=["src", "tgt"])
    dev_dataset   = records_to_dataset(make_records(dev_df)).map(preprocess, remove_columns=["src", "tgt"])
    test_dataset  = records_to_dataset(make_records(test_df)).map(preprocess, remove_columns=["src", "tgt"])
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

    print(f"Train: {len(train_dataset)}  Dev: {len(dev_dataset)}  Test: {len(test_dataset)}")

    # ── Train ──
    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir),
        per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=5e-4,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        logging_steps=50,
        report_to="none",
        optim="adafactor",
        save_total_limit=1,
        seed=RANDOM_SEED,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
    )
    gc.collect(); torch.cuda.empty_cache()
    trainer.train()

    print("\nTraining log:")
    for entry in trainer.state.log_history:
        print(entry)

    # ── Clean up training dir to free disk BEFORE saving ──
    for p in run_dir.glob("checkpoint-*"):
        shutil.rmtree(p, ignore_errors=True)
    shutil.rmtree(run_dir, ignore_errors=True)
    print(f"Training dir cleaned: {run_dir}")

    # ── Save best model + tokenizer (fp16 to save disk) ──
    save_dir = OUT_DIR / f"best-{out_suffix}"
    model.half()
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    model.float()  # back to fp32 for eval
    print(f"Model saved (fp16) to {save_dir}")

    # ── Evaluate on test set with sacrebleu ──
    gc.collect(); torch.cuda.empty_cache()

    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    sources    = [row[src_col] for _, row in test_df.iterrows()]
    references = [row[tgt_col] for _, row in test_df.iterrows()]
    decoded_preds = []

    tokenizer.src_lang = src_lang
    for i in range(0, len(sources), 8):
        batch = sources[i:i+8]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                max_new_tokens=MAX_LEN, num_beams=4,
                                no_repeat_ngram_size=3)
        decoded_preds += tokenizer.batch_decode(out, skip_special_tokens=True)

    chrf2pp = sacrebleu.corpus_chrf(decoded_preds, [references], word_order=2, beta=2)
    chrf2   = sacrebleu.corpus_chrf(decoded_preds, [references], word_order=0, beta=2)
    chrfpp  = sacrebleu.corpus_chrf(decoded_preds, [references], word_order=2, beta=1)
    chrf    = sacrebleu.corpus_chrf(decoded_preds, [references], word_order=0, beta=1)
    bleu    = sacrebleu.corpus_bleu(decoded_preds, [references])

    metrics = {
        "direction": direction_label,
        "BLEU": round(bleu.score, 2),
        "ChrF": round(chrf.score, 2),
        "ChrF++": round(chrfpp.score, 2),
        "ChrF2": round(chrf2.score, 2),
        "ChrF2++": round(chrf2pp.score, 2),
    }
    print(f"\n=== {direction_label} Test Metrics ===")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    # ── Samples ──
    print(f"\n=== {direction_label} Sample Translations ===")
    for i in range(min(5, len(sources))):
        print(f"\n  SRC : {sources[i]}")
        print(f"  REF : {references[i]}")
        print(f"  HYP : {decoded_preds[i]}")

    # ── Cleanup: delete model + trainer, free GPU + disk ──
    del model, trainer, tokenizer, train_dataset, dev_dataset, test_dataset
    gc.collect()
    torch.cuda.empty_cache()
    # Clear HF cache to free disk for next direction
    hf_cache = Path.home() / ".cache" / "huggingface"
    if hf_cache.exists():
        shutil.rmtree(hf_cache, ignore_errors=True)
        print("HF cache cleared")
    print(f"\nGPU freed. Remaining: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

    return metrics


In [10]:
# ── Cell 10: QOM → ES ─────────────────────────────────────────────────────────
metrics_qom2es = run_direction(
    direction_label="QOM → ES",
    src_col="qom", tgt_col="es",
    src_lang=TGT_LANG, tgt_lang=SRC_LANG,
    train_df=train_df, dev_df=dev_df, test_df=test_df,
    out_suffix="qom2es",
)



  DIRECTION: QOM → ES



tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training log:
{'loss': 26.51082763671875, 'grad_norm': 14.580694198608398, 'learning_rate': 0.0004985381861575179, 'epoch': 0.029848518767256176, 'step': 50}
{'loss': 23.33524169921875, 'grad_norm': 15.663052558898926, 'learning_rate': 0.000497046539379475, 'epoch': 0.05969703753451235, 'step': 100}
{'loss': 21.25977783203125, 'grad_norm': 11.61025333404541, 'learning_rate': 0.0004955548926014319, 'epoch': 0.08954555630176853, 'step': 150}
{'loss': 20.4850048828125, 'grad_norm': 15.213932037353516, 'learning_rate': 0.0004941229116945107, 'epoch': 0.1193940750690247, 'step': 200}
{'loss': 18.988607177734377, 'grad_norm': 15.354308128356934, 'learning_rate': 0.0004926312649164678, 'epoch': 0.14924259383628088, 'step': 250}
{'loss': 19.012176513671875, 'grad_norm': 14.590439796447754, 'learning_rate': 0.0004911396181384248, 'epoch': 0.17909111260353705, 'step': 300}
{'loss': 18.27052978515625, 'grad_norm': 18.550060272216797, 'learning_rate': 0.0004896479713603818, 'epoch': 0.20893963137

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved (fp16) to /kaggle/working/qom-nllb/best-qom2es

=== QOM → ES Test Metrics ===
  direction: QOM → ES
  BLEU: 24.73
  ChrF: 47.86
  ChrF++: 46.42
  ChrF2: 46.71
  ChrF2++: 45.4

=== QOM → ES Sample Translations ===

  SRC : Am da 'avia'aco anqo'onaxama
  REF : Si vas a pescar
  HYP : Cuando vos vayas, debes ir

  SRC : so tala'
  REF : al río
  HYP : al río

  SRC : nache da huo'o ca oxonec
  REF : Y si obtenés una presa,
  HYP : y cuando hay un vencedor

  SRC : nache que'eca 'auauatton da qanallec
  REF : que vos sabés que es comestible,
  HYP : entonces esa (la comida) que conocés para comer

  SRC : nache 'andouo qaq que'eca auauatton da sa qanallec
  REF : entonces traelo a casa y lo que vos sabés que no es comestible
  HYP : entonces tenés que traerlo y esa cosa que conocés no se come.
HF cache cleared

GPU freed. Remaining: 9.8 GB


In [11]:
# ── Cell 11: Results summary ──────────────────────────────────────────────────
results = pd.DataFrame([metrics_qom2es])
print("\n" + "="*70)
print("  FINAL RESULTS — Random Splits")
print("="*70)
print(results.to_string(index=False))
print()
print(f"Train: {len(train_df)}  Dev: {len(dev_df)}  Test: {len(test_df)}")
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {NUM_EPOCHS}  Effective batch: {TRAIN_BATCH * GRAD_ACCUM}")
print(f"View: {TRAIN_VIEW}  Split: global random")
print(f"Saved models: {list(OUT_DIR.glob('best-*'))}")



  FINAL RESULTS — Random Splits
direction  BLEU  ChrF  ChrF++  ChrF2  ChrF2++
 QOM → ES 24.73 47.86   46.42  46.71     45.4

Train: 26802  Dev: 3256  Test: 3273
Model: facebook/nllb-200-distilled-600M
Epochs: 10  Effective batch: 16
View: line  Split: global random
Saved models: [PosixPath('/kaggle/working/qom-nllb/best-qom2es')]
